# Mirror Param Calculation
## Consts:

In [2]:
cameraAlpha: int = 20 # half angle of camera - deg
mirrorHeightFromGround: int = 180 # mm
fieldRadius: int = 2500 # mm

hyperbolicC: float = 25 # mm

amountOfPoints: int = 500

## Calculations:

In [8]:
import math

beta = math.degrees(math.atan(fieldRadius / mirrorHeightFromGround))

class Calculations:
    @staticmethod
    def cot(deg: float) -> float:
        return 1 / math.tan(math.radians(deg))

    @staticmethod
    def mirrorRadius(c: float) -> float:
        return 2 * c / (Calculations.cot(cameraAlpha) + Calculations.cot(beta))

    @staticmethod
    def mirrorZmax(c: float) -> float:
        return c - Calculations.mirrorRadius(c) * Calculations.cot(beta)

    @staticmethod
    def hyperbolicA(c: float) -> float:
        quadraticEquationB = (c ** 2 + Calculations.mirrorZmax(c) ** 2 + Calculations.mirrorRadius(c) ** 2)
        quadraticEquationC = (Calculations.mirrorZmax(c) ** 2) * (c ** 2)
        return math.sqrt((quadraticEquationB - math.sqrt(quadraticEquationB ** 2 - 4 * quadraticEquationC)) / 2)

    @staticmethod
    def hyperbolicB(c: float) -> float:
        return math.sqrt(c ** 2 - Calculations.hyperbolicA(c) ** 2)

    @staticmethod
    def mirrorHeight(c: float) -> float:
        return Calculations.mirrorZmax(c) - Calculations.hyperbolicA(c)


## Run

In [25]:
print(Calculations.hyperbolicA(hyperbolicC), Calculations.hyperbolicB(hyperbolicC))

17.035211360973516 18.29758382647717


# Hyperbolic Function Drawing in DXF
## Hyperbolic Function:

In [4]:
def hyperbolicFunction(x: float, a: float, b: float) -> float: #returns y
    return (a * math.sqrt(1 + (x ** 2) / (b ** 2))) - a

## Points Calculations

In [15]:
radius = Calculations.mirrorRadius(hyperbolicC)
hyperbolicA = Calculations.hyperbolicA(hyperbolicC)
hyperbolicB = Calculations.hyperbolicB(hyperbolicC)

pointsX = [radius * ((i / amountOfPoints) ** 1.5) for i in range(0, amountOfPoints + 1)]
pointsY = [hyperbolicFunction(point, hyperbolicA, hyperbolicB) for point in pointsX]

points = [(pointX, pointY) for pointX, pointY in zip(pointsX, pointsY)]
F2 = (0, Calculations.mirrorHeight(hyperbolicC))
max = (Calculations.mirrorRadius(hyperbolicC), Calculations.mirrorHeight(hyperbolicC))

In [13]:
print(points)

[(0.0, 0.0), (0.0015861577482910415, 6.400630780944994e-08), (0.004486331599392722, 5.120504731337405e-07), (0.008241917426577391, 1.7281702966442936e-06), (0.012689261986328332, 4.096403376507851e-06), (0.017733782741083847, 8.000786930750792e-06), (0.023311662809249806, 1.3825357452645903e-05), (0.02937605259467397, 2.1954150351888302e-05), (0.035890652795141774, 3.277119945011009e-05), (0.04282625920385811, 4.666053644086787e-05), (0.05015871212923739, 6.400619023949616e-05), (0.05786759120336005, 8.519218623348479e-05), (0.06593533941261913, 0.00011060254546890746), (0.07434665020169208, 0.00014062128374092708), (0.08308802397674656, 0.00017563241058837775), (0.09214743815383589, 0.00021601992823860883), (0.10151409589062665, 0.00026216783039600955), (0.11117823089662919, 0.0003144601010411918), (0.12113095318360347, 0.000373280713031221), (0.1313641253329788, 0.00043901362671761035), (0.14187026192867078, 0.0005120427884151013), (0.1526424468593726, 0.0005927521287922843), (0.1636

## DXF Export

In [ ]:
!pip install ezdxf

In [16]:
import ezdxf
from ezdxf.units import MM

doc = ezdxf.new(setup=True)
doc.units = MM
msp = doc.modelspace()

layerName: str = "MIRROR_PROFILE"

if layerName not in doc.layers:
    doc.layers.add(layerName)

points3D = [(x, y, 0.0) for x, y in points]

msp.add_spline(fit_points=points3D, dxfattribs={"layer": layerName})
msp.add_line((0, 0), F2, dxfattribs={"layer": layerName})
msp.add_line(max, F2, dxfattribs={"layer": layerName})

doc.saveas("mirror.dxf")
print("Finished exporting mirror profile")

Finished exporting mirror profile
